# 🔍 Run Data Quality Checks

**Purpose.** Run every assertion defined in `notebooks/utils/data_quality_checks` against the live Bronze, Silver, and Gold tables, and produce a single pass/fail report.

**When to run.** After any pipeline run (Bronze → Silver → Gold). The whole suite completes in under a minute.

**Output.** A summary table showing every check, whether it passed, and a one-line message. A non-zero failure count is the trigger to investigate.

In [0]:
%run ../utils/data_quality_checks


In [0]:
# All assertion functions, grouped by layer.
CHECKS = {
    "Bronze": [
        ("bronze_table_not_empty",         bronze_table_not_empty),
        ("bronze_audit_columns_populated", bronze_audit_columns_populated),
        ("bronze_source_files_expected",   bronze_source_files_expected),
    ],
    "Silver": [
        ("silver_row_count_sensible",       silver_row_count_sensible),
        ("silver_no_negative_fares",        silver_no_negative_fares),
        ("silver_no_zero_distance",         silver_no_zero_distance),
        ("silver_dropoff_after_pickup",     silver_dropoff_after_pickup),
        ("silver_pickup_in_expected_window",silver_pickup_in_expected_window),
        ("silver_zones_populated",          silver_zones_populated),
    ],
    "Gold": [
        ("gold_daily_revenue_reconciles",     gold_daily_revenue_reconciles),
        ("gold_hourly_demand_is_168",         gold_hourly_demand_is_168),
        ("gold_top_zones_count",              gold_top_zones_count),
        ("gold_payment_breakdown_reconciles", gold_payment_breakdown_reconciles),
    ],
}

results = []
total_passed = 0
total_failed = 0

for layer, checks in CHECKS.items():
    for name, fn in checks:
        try:
            passed, message = fn(spark)
        except Exception as e:
            passed, message = False, f"Exception raised: {type(e).__name__}: {e}"
        results.append((layer, name, passed, message))
        if passed:
            total_passed += 1
        else:
            total_failed += 1

# Print a clean text report.
print("=" * 88)
print(f"Data Quality Report")
print("=" * 88)
for layer, name, passed, message in results:
    icon = "✓" if passed else "✗"
    print(f"  [{icon}] {layer:<7} {name:<40}  {message}")
print("=" * 88)
print(f"  Passed: {total_passed}/{total_passed + total_failed}")
print(f"  Failed: {total_failed}/{total_passed + total_failed}")
print("=" * 88)

In [0]:
from pyspark.sql import Row

results_df = spark.createDataFrame([
    Row(layer=layer, check=name, passed=passed, message=message)
    for layer, name, passed, message in results
])

display(results_df.orderBy("layer", "check"))